In [18]:
import sys
import os

sys.path.append(os.path.abspath('..'))

In [19]:

import pandas as pd
import numpy as  np
import datetime as dt
import pandas as pd
import numpy as np
import datetime as dt

from Python_arq import engines as engs
from sqlalchemy import text
from calendar import day_name
from sqlalchemy import text
from pathlib import Path


eng = engs.get_engine()
text_qry = text(engs.load_qry('qry_olos.sql'))
df = pd.read_sql(text_qry, eng)


In [20]:
dia_semana = {
    'Monday': 'Segunda',
    'Tuesday': 'Terça',
    'Wednesday': 'Quarta',
    'Thursday': 'Quinta',
    'Friday': 'Sexta',
    'Saturday': 'Sábado',
    'Sunday': 'Domingo'
}

df['day_week'] = df['data'].apply(lambda x: day_name[x.weekday()]).map(dia_semana)
df['data'] = pd.to_datetime(df['data'])
df['semana_mes'] = (
    df['data'].dt.day.sub(1) // 7 + 1 
)
df['day_week'] = df['day_week'] + '_W' + df['semana_mes'].astype(str)

In [21]:
## performance bases | Hoje ## 
df_today = df[df['data'].dt.date == dt.datetime.today().date()]
df_today = df_today.groupby(['area','base_type','tablename']).agg(
    Total_tentativas = ('tentativas','sum'),
    total_atendidas = ('atendidas','sum')
).reset_index()
df_today['performance'] = (df_today['total_atendidas'] / df_today['Total_tentativas'] * 100).round(2)
df_today = df_today.sort_values('performance', ascending=False)

In [22]:
## performance bases por id de campanha | Hoje ## 
df_today = df[df['campaign_id'] == '1299']
df_today = df[df['data'].dt.date == dt.datetime.today().date()]
df_today = df_today.groupby(['area','base_type']).agg(
    Total_tentativas = ('tentativas','sum'),
    total_atendidas = ('atendidas','sum')
).reset_index()
df_today['performance'] = (df_today['total_atendidas'] / df_today['Total_tentativas'] * 100).round(2)
df_today = df_today.sort_values('performance', ascending=False)
df_today

,area,base_type,Total_tentativas,total_atendidas,performance
4,Psicologia,Material Rico,602,22,3.65
2,Fisioterapia,evento,220,8,3.64
5,Psicologia,ativa,1470,44,2.99
3,Fisioterapia,inativa,1481,39,2.63
1,Fisioterapia,ativa,881,18,2.04
0,Enfermagem,inativa,1330,26,1.95
6,Psicologia,inativa,766,14,1.83
7,Psiquiatria,evento,213,2,0.94


In [23]:
## top10 bases ## 
df_top10 = df.groupby(['area','base_type']).agg(
    total_tentativas = ('tentativas','sum'),
    total_atendidas = ('atendidas','sum')
).reset_index()
df_top10['performance'] = (df_top10['total_atendidas'] / df_top10['total_tentativas'] * 100).round(2)
df_top10 = df_top10.sort_values('performance', ascending=False).head(10)
df_top10

,area,base_type,total_tentativas,total_atendidas,performance
57,Veterinária,ativa,58,6,10.34
25,Multi,Carrinho,66,5,7.58
55,Veterinária,Base Leads,4413,256,5.80
60,Veterinária,legado,167,9,5.39
30,Multi,legado,64,3,4.69
31,Nutrição,Base Leads,5716,257,4.50
28,Multi,evento,10897,467,4.29
26,Multi,Material Rico,3556,149,4.19
36,Pediatria,Base Leads,3153,131,4.15
40,Pediatria,evento,4805,189,3.93


In [25]:
## localizacao por campanha

df_camp = df.groupby(['campaign_id','tablename']).agg(
    tentativas = ('tentativas','sum'),
    atendidas = ('atendidas','sum')
).reset_index()
df_camp['performance'] = (df_camp['atendidas'] / df_camp['tentativas'] * 100).round(2)
df_camp

,campaign_id,tablename,tentativas,atendidas,performance
0,1025,,61,26,42.62
1,1025,SECAD_1025_NUTRICAO_GROWTH_0_BASELEADSPT01_121...,555,40,7.21
2,1025,SECAD_1025_NUTRICAO_GROWTH_0_BASELEADSPT02_121...,315,17,5.40
3,1025,SECAD_1025_NUTRICAO_GROWTH_0_BASELEADSPT03_121...,665,34,5.11
4,1025,SECAD_1025_NUTRICAO_GROWTH_0_BASELEADSPT04_121...,592,35,5.91
...,...,...,...,...,...
1177,1605,_1605__1605_FISIOTERAPIA_PAGEVIEW_06042026_V50...,1704,28,1.64
1178,1605,_1605__1605_FISIOTERAPIA_PAGEVIEW_06042026_V50...,1282,27,2.11
1179,1605,_1605__1605_FISIOTERAPIA_PAGEVIEW_06042026_V50...,813,12,1.48
1180,1605,_1605__1605_FISIOTERAPIA_PAGEVIEW_06042026_V50...,1331,25,1.88
